This notebook matches the sampled/textual Alibaba Disk rows back to their original numerical SMART-log rows.


In [ ]:
from pathlib import Path
from datetime import datetime
import logging
import re

import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
# =========================
# Repository-relative paths
# =========================
PROJECT_ROOT = Path.cwd()

INPUT_ROOT = PROJECT_ROOT / "input"
OUTPUT_ROOT = PROJECT_ROOT / "output"

ALIBABA_RAW_DIR = INPUT_ROOT / "alibaba_disk" / "original_csv"

# Alibaba Disk uses anonymized numerical model IDs.
# Change MODEL_VALUE to 2 to process model-two.
MODEL_VALUE = 1
MODEL_NAME_MAP = {
    1: "model-one",
    2: "model-two",
}
DISK_MODEL_ID = MODEL_NAME_MAP[MODEL_VALUE]

OUTPUT_DATA_DIR = OUTPUT_ROOT / "data" / DISK_MODEL_ID
SAMPLED_TEXTUAL_DIR = OUTPUT_DATA_DIR / "monthly_textual"
MATCHED_MONTHLY_DIR = OUTPUT_DATA_DIR / "monthly_numerical"

MATCHED_MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

print(f"Disk model id       : {DISK_MODEL_ID}")
print(f"Alibaba raw dir     : {ALIBABA_RAW_DIR}")
print(f"Sampled textual dir : {SAMPLED_TEXTUAL_DIR}")
print(f"Matched output dir  : {MATCHED_MONTHLY_DIR}")


In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


## Helper Functions


In [ ]:
def standardize_date_series(series: pd.Series) -> pd.Series:
    """Convert a date-like series to YYYY-MM-DD string format."""
    return pd.to_datetime(series, errors="coerce").dt.strftime("%Y-%m-%d")


def normalize_model_value(value) -> str:
    """Normalize model values for matching."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text in {"model-one", "MA1"}:
        return "1"
    if text in {"model-two", "MB1"}:
        return "2"
    if text.endswith(".0"):
        text = text[:-2]
    return text


def parse_months_from_textual_file(label_path: Path) -> list[str]:
    """Return all YYYYMM months covered by a sampled textual CSV."""
    df_dates = pd.read_csv(label_path, usecols=["date"])
    dates = pd.to_datetime(df_dates["date"], errors="coerce").dropna()
    months = sorted(dates.dt.strftime("%Y%m").unique())
    return months


def raw_path_for_month(month_key: str) -> Path:
    """Return the Alibaba monthly SMART-log path for YYYYMM."""
    return ALIBABA_RAW_DIR / f"smartlog_data_{month_key}.csv"


def load_sampled_keys(label_path: Path) -> pd.DataFrame:
    """Load matching keys and labels from a sampled textual CSV."""
    label_df = pd.read_csv(label_path, low_memory=False)

    required = {"date", "serial_number"}
    missing = required - set(label_df.columns)
    if missing:
        raise ValueError(f"{label_path.name} is missing required columns: {sorted(missing)}")

    label_df = label_df.copy()
    label_df["date"] = standardize_date_series(label_df["date"])
    label_df["serial_number"] = label_df["serial_number"].astype(str).str.strip()

    if "model" in label_df.columns:
        label_df["model_key"] = label_df["model"].map(normalize_model_value)
    else:
        label_df["model_key"] = str(MODEL_VALUE)

    # Keep only rows for the selected Alibaba model.
    label_df = label_df[label_df["model_key"] == str(MODEL_VALUE)].copy()

    # Build or repair failure_7d.
    if "failure_7d" in label_df.columns:
        failure_7d = pd.to_numeric(label_df["failure_7d"], errors="coerce").fillna(0).astype(int)
    elif "upcoming_failure" in label_df.columns:
        failure_7d = label_df["upcoming_failure"].astype(str).str.startswith("Yes").astype(int)
    else:
        failure_7d = pd.Series(0, index=label_df.index, dtype=int)

    if "failure" in label_df.columns:
        current_failure = pd.to_numeric(label_df["failure"], errors="coerce").fillna(0).astype(int)
        failure_7d = ((failure_7d == 1) | (current_failure == 1)).astype(int)

    label_df["failure_7d"] = failure_7d

    key_cols = ["date", "serial_number", "model_key", "failure_7d"]
    label_df = label_df[key_cols].dropna(subset=["date", "serial_number"]).drop_duplicates()

    return label_df


def load_raw_month(month_key: str) -> pd.DataFrame:
    """Load and standardize one Alibaba monthly raw SMART-log file."""
    raw_path = raw_path_for_month(month_key)
    if not raw_path.is_file():
        raise FileNotFoundError(f"Raw monthly file not found: {raw_path}")

    raw_df = pd.read_csv(raw_path, low_memory=False)

    if "date" in raw_df.columns:
        raw_df["date"] = standardize_date_series(raw_df["date"])
    elif "dt" in raw_df.columns:
        raw_df["date"] = pd.to_datetime(
            raw_df["dt"].astype(str),
            format="%Y%m%d",
            errors="coerce",
        ).dt.strftime("%Y-%m-%d")
    else:
        raise ValueError(f"{raw_path.name} has neither 'date' nor 'dt' column.")

    if "serial_number" not in raw_df.columns:
        raise ValueError(f"{raw_path.name} is missing 'serial_number'.")

    raw_df["serial_number"] = raw_df["serial_number"].astype(str).str.strip()

    if "model" in raw_df.columns:
        raw_df["model_key"] = raw_df["model"].map(normalize_model_value)
        raw_df = raw_df[raw_df["model_key"] == str(MODEL_VALUE)].copy()
    else:
        raw_df["model"] = MODEL_VALUE
        raw_df["model_key"] = str(MODEL_VALUE)

    return raw_df


## Match Sampled Textual Rows Back to Raw Monthly SMART Logs


In [ ]:
def process_one_textual_file(label_path: Path) -> int:
    """Match one sampled textual CSV back to Alibaba raw monthly SMART logs."""
    label_keys = load_sampled_keys(label_path)
    if label_keys.empty:
        logging.info(f"{label_path.name} -> no sampled rows for {DISK_MODEL_ID}")
        return 0

    months = parse_months_from_textual_file(label_path)
    matched_parts = []

    for month_key in months:
        try:
            raw_df = load_raw_month(month_key)
        except FileNotFoundError:
            logging.warning(f"{label_path.name} -> missing raw file for {month_key}")
            continue

        month_label_keys = label_keys[
            pd.to_datetime(label_keys["date"], errors="coerce").dt.strftime("%Y%m") == month_key
        ].copy()

        if month_label_keys.empty:
            continue

        matched = raw_df.merge(
            month_label_keys,
            on=["date", "serial_number", "model_key"],
            how="inner",
            suffixes=("", "_label"),
        )

        if matched.empty:
            continue

        # If the raw file already has a current failure column, make sure current failures are positive.
        if "failure" in matched.columns:
            current_failure = pd.to_numeric(matched["failure"], errors="coerce").fillna(0).astype(int)
            matched["failure_7d"] = (
                (pd.to_numeric(matched["failure_7d"], errors="coerce").fillna(0).astype(int) == 1)
                | (current_failure == 1)
            ).astype(int)

        matched = matched.drop(columns=["model_key"], errors="ignore")
        matched_parts.append((month_key, matched))

    if not matched_parts:
        logging.info(f"{label_path.name} -> no matched rows")
        return 0

    total_rows = 0
    for month_key, matched in matched_parts:
        month_name = f"{month_key[:4]}-{month_key[4:]}"
        output_path = MATCHED_MONTHLY_DIR / f"matched_raw_{month_name}.csv"

        if output_path.exists():
            previous = pd.read_csv(output_path, low_memory=False)
            matched = pd.concat([previous, matched], ignore_index=True)

        matched = matched.drop_duplicates(subset=["date", "serial_number"], keep="first")
        matched.to_csv(output_path, index=False)

        total_rows += len(matched)
        logging.info(f"{label_path.name} -> {len(matched):,} rows -> {output_path.name}")

    return total_rows


In [ ]:
textual_files = sorted(SAMPLED_TEXTUAL_DIR.glob("*.csv"))

if not textual_files:
    raise FileNotFoundError(f"No sampled textual CSV files found under: {SAMPLED_TEXTUAL_DIR}")

print(f"Found {len(textual_files)} sampled textual CSV files.")

total_matched = 0
for label_path in tqdm(textual_files, desc=f"Matching {DISK_MODEL_ID}"):
    total_matched += process_one_textual_file(label_path)

print(f"Total matched rows: {total_matched:,}")


## Save Numerical Train/Validation/Test Files


In [ ]:
matched_files = sorted(MATCHED_MONTHLY_DIR.glob("matched_raw_*.csv"))

if not matched_files:
    raise FileNotFoundError(f"No matched numerical CSV files found under: {MATCHED_MONTHLY_DIR}")

df = pd.concat(
    [pd.read_csv(path, low_memory=False) for path in matched_files],
    ignore_index=True,
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date", "serial_number"]).copy()
df = df.sort_values(["date", "serial_number"]).reset_index(drop=True)

if "failure_7d" not in df.columns:
    raise ValueError("Matched numerical data does not contain failure_7d.")

# Ensure current failures are also positive.
if "failure" in df.columns:
    current_failure = pd.to_numeric(df["failure"], errors="coerce").fillna(0).astype(int)
    label_failure = pd.to_numeric(df["failure_7d"], errors="coerce").fillna(0).astype(int)
    df["failure_7d"] = ((label_failure == 1) | (current_failure == 1)).astype(int)
else:
    df["failure_7d"] = pd.to_numeric(df["failure_7d"], errors="coerce").fillna(0).astype(int)

min_date = df["date"].min().normalize()

# Alibaba Disk protocol:
# - train: first 15 months, used as the development/training period
# - val  : last 3 months within the first 15 months, used for threshold calibration
# - test : later months, held out for final inference/evaluation
TRAIN_START = min_date
TRAIN_END = TRAIN_START + pd.DateOffset(months=15) - pd.Timedelta(days=1)

VAL_START = TRAIN_END - pd.DateOffset(months=3) + pd.Timedelta(days=1)
VAL_END = TRAIN_END

TEST_START = TRAIN_END + pd.Timedelta(days=1)
TEST_END = df["date"].max().normalize()

train_df = df[
    (df["date"] >= TRAIN_START)
    & (df["date"] <= TRAIN_END)
].copy()

val_df = df[
    (df["date"] >= VAL_START)
    & (df["date"] <= VAL_END)
].copy()

test_df = df[
    (df["date"] >= TEST_START)
    & (df["date"] <= TEST_END)
].copy()

if train_df.empty:
    raise ValueError(f"Empty train split: {TRAIN_START.date()} to {TRAIN_END.date()}")

if val_df.empty:
    raise ValueError(f"Empty validation split: {VAL_START.date()} to {VAL_END.date()}")

if test_df.empty:
    raise ValueError(f"Empty test split: {TEST_START.date()} to {TEST_END.date()}")

# Save dates as YYYY-MM-DD strings.
for split_df in [train_df, val_df, test_df]:
    split_df["date"] = split_df["date"].dt.strftime("%Y-%m-%d")

train_df.to_csv(OUTPUT_DATA_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DATA_DIR / "val.csv", index=False)
test_df.to_csv(OUTPUT_DATA_DIR / "test.csv", index=False)

print(f"Saved train.csv: {train_df.shape} [{TRAIN_START.date()} to {TRAIN_END.date()}]")
print(f"Saved val.csv  : {val_df.shape} [{VAL_START.date()} to {VAL_END.date()}]")
print(f"Saved test.csv : {test_df.shape} [{TEST_START.date()} to {TEST_END.date()}]")
